# Python 与 PDF

在现代办公环境中，PDF 文件因其跨平台兼容性和丰富的功能而被广泛使用。无论是文档共享、报告撰写还是数据分析，PDF 文件都扮演着重要角色。然而，处理 PDF 文件时常常需要进行多种操作，如合并、拆分、提取文本和表格数据、加密和解密等。Python 提供了多个强大的库来处理这些任务，其中 PyPDF2 和 pdfplumber 是两个非常实用的工具。

## 1. 使用 PyPDF2 和 pdfplumber 操作 PDF 文件

### 1.1 课程目标
- 了解 PyPDF2 和 pdfplumber 库的基本功能和应用场景
- 掌握使用 PyPDF2 进行 PDF 文件的合并、拆分、加密、解密等操作
- 学会使用 pdfplumber 提取 PDF 中的文本和表格数据
- 能够结合这两个库完成 PDF 文件的读取、处理和分析任务


### 1.2 PyPDF2 和 pdfplumber 简介

**PyPDF2**

PyPDF2 是一个纯 Python 库，用于读取和写入 PDF 文件。它支持多种 PDF 操作，包括但不限于：
- 合并 PDF 文件
- 拆分 PDF 页面
- 旋转页面
- 裁剪页面
- 添加水印
- 加密和解密 PDF

**pdfplumber**  
pdfplumber 是一个用于从 PDF 文件中提取文本和表格的库。它提供了简单直观的 API 来访问 PDF 文件的内容，支持：
- 提取文本
- 提取表格
- 提取元数据
- 访问页面内容

### 1.3 环境准备

In [ ]:
!pip install PyPDF2 pdfplumber

!pip install reportlab #用于创建pdf文件


### 1.4 基础操作示例


#### 1.4.1 创建 PDF 文件
我们在 SamplePDFFiles 中创建五个 pdf 文件，前三个为文字pdf文件，第四个为表格 pdf 文件，第五个为水印 pdf 文件。


In [ ]:
import os
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle
from reportlab.lib import colors

# 定义创建 PDF 文件的函数
def create_sample_pdf(file_name, content, table_data=None):
    # 创建一个 PDF 画布
    c = canvas.Canvas(file_name, pagesize=letter)
    width, height = letter  # 获取页面宽度和高度

    # 添加标题
    c.setFont("Helvetica-Bold", 16)
    c.drawCentredString(width / 2.0, height - 72, "Sample PDF Document")

    # 添加内容
    c.setFont("Helvetica", 12)
    c.drawString(72, height - 144, content)

    # 如果提供了表格数据，添加表格
    if table_data:
        # 创建一个 PDF 文档
        doc = SimpleDocTemplate(file_name, pagesize=letter)
        elements = []

        # 创建表格
        table = Table(table_data)
        table.setStyle(TableStyle([
            ('BACKGROUND', (0, 0), (-1, 0), colors.grey),
            ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
            ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
            ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
            ('BOTTOMPADDING', (0, 0), (-1, 0), 12),
            ('BACKGROUND', (0, 1), (-1, -1), colors.beige),
            ('GRID', (0, 0), (-1, -1), 1, colors.black),
        ]))

        elements.append(table)
        doc.build(elements)
    else:
        # 保存 PDF 文件
        c.save()

# 创建文件夹
folder_name = "SamplePDFFiles"
os.makedirs(folder_name, exist_ok=True)

# 创建四个示例 PDF 文件
for i in range(1, 5):
    file_name = os.path.join(folder_name, f"document_{i}.pdf")
    content = f"This is the content of document {i}. It includes some English text."
    
    # 为第四个文件添加表格
    if i == 4:
        table_data = [
            ['Header 1', 'Header 2', 'Header 3'],
            ['Row 1, Col 1', 'Row 1, Col 2', 'Row 1, Col 3'],
            ['Row 2, Col 1', 'Row 2, Col 2', 'Row 2, Col 3'],
            ['Row 3, Col 1', 'Row 3, Col 2', 'Row 3, Col 3']
        ]
        create_sample_pdf(file_name, content, table_data)
    else:
        create_sample_pdf(file_name, content)
    
    print(f"Sample PDF file created: {file_name}")

#### 1.4.2 使用 PyPDF2 合并 PDF 文件

In [ ]:
import PyPDF2
from PyPDF2 import PdfReader, PdfWriter
import os

# 定义 PDF 文件所在文件夹路径
pdf_folder = "SamplePDFFiles"

# 创建一个 PDF 写入对象
output = PdfWriter()

# 读取多个 PDF 文件并添加到输出对象
for i in range(1, 5):  # 从 1 到 4
    file_path = os.path.join(pdf_folder, f'document_{i}.pdf')
    if os.path.exists(file_path):
        with open(file_path, 'rb') as file:
            reader = PdfReader(file)
            for page_num in range(len(reader.pages)):
                output.add_page(reader.pages[page_num])
    else:
        print(f"文件 {file_path} 不存在，跳过。")

# 写入合并后的 PDF 文件
merged_file_path = 'merged_document.pdf'
with open(merged_file_path, 'wb') as file:
    output.write(file)

print(f"PDF 文件合并完成！合并后的文件已保存为：{merged_file_path}")

#### 1.4.2 使用 PyPDF2 拆分 PDF 文件

In [ ]:
import os
from PyPDF2 import PdfReader, PdfWriter

# 定义输出文件夹路径
output_folder = "拆分后的PDF文件"
os.makedirs(output_folder, exist_ok=True)  # 确保文件夹存在

# 读取 PDF 文件
with open('merged_document.pdf', 'rb') as file:
    reader = PdfReader(file)
    writer = PdfWriter()

    # 拆分每一页
    for i in range(len(reader.pages)):
        writer.add_page(reader.pages[i])
        
        # 写入拆分后的 PDF 文件
        output_file_path = os.path.join(output_folder, f'page_{i+1}.pdf')
        with open(output_file_path, 'wb') as output_file:
            writer.write(output_file)
        writer = PdfWriter()  # 重置 writer 以准备下一页

print(f"PDF 文件已拆分，每一页保存在文件夹 '{output_folder}' 中。")

#### 1.4.3 使用 pdfplumber 提取 PDF 中的文本

In [ ]:
import pdfplumber

# 打开 PDF 文件
with pdfplumber.open("merged_document.pdf") as pdf:
    # 遍历每一页
    for page_num, page in enumerate(pdf.pages, start=1):
        # 提取文本
        text = page.extract_text()
        if text:  # 确保页面有文本内容
            print(f"Page {page_num}:\n{text}\n")
        else:
            print(f"Page {page_num} has no text content.")

#### 1.4.4  使用 pdfplumber 提取 PDF 中的表格

In [ ]:
import pdfplumber

with pdfplumber.open("merged_document.pdf") as pdf:
    page = pdf.pages[3]
    table = page.extract_table()

table

### 1.4 应用场景演示


**批量提取 PDF 表格数据**

需求：从一系列 PDF 文件中提取表格数据，用于数据分析。

In [ ]:
import pdfplumber
import pandas as pd
import os

# 假设所有 PDF 文件都在这个文件夹中
folder_path = 'SamplePDFFiles'
output_data = []

for filename in os.listdir(folder_path):
    if filename.endswith('.pdf'):
        with pdfplumber.open(os.path.join(folder_path, filename)) as pdf:
            for page in pdf.pages:
                table = page.extract_table()
                if table:
                    output_data.append(table)

# 将提取的数据转换为 DataFrame
df = pd.DataFrame(output_data)
df


## 2. 合并、拆分与加密 PDF 文件

### 2.1 课程目标

- 了解如何使用 PyPDF2 对 PDF 文件进行加密和解密

### 2.2 加密 PDF 文件

#### 步骤 1: 读取 PDF 文件并设置密码

设置使用者的密码为`user`，拥有者的密码为`owner`。

In [ ]:
# 打开 PDF 文件
with open('merged_document.pdf', 'rb') as file:
    reader = PdfReader(file)
    writer = PdfWriter()

    # 遍历所有页面并添加到新的 PDFWriter 对象中
    for page in reader.pages:
        writer.add_page(page)

    # 设置 PDF 密码
    writer.encrypt(user_pwd='user', owner_pwd='owner', use_128bit=True)



#### 步骤 2: 写入加密后的 PDF 文件

In [ ]:
# 保存加密后的 PDF 文件
with open('encrypted_merged_document.pdf', 'wb') as file:
    writer.write(file)

print("Encrypted PDF created: encrypted_merged_document.pdf")

删除本次实训所有生成的文件。

In [ ]:
import os
import shutil

# 删除生成的 PDF 文件
files_to_delete = [
    'merged_document.pdf',
    'encrypted_merged_document.pdf'
]

for file in files_to_delete:
    if os.path.exists(file):
        os.remove(file)
        print(f"Deleted file: {file}")

# 删除生成的拆分后的 PDF 文件夹
split_folder = "拆分后的PDF文件"
if os.path.exists(split_folder):
    shutil.rmtree(split_folder)
    print(f"Deleted folder: {split_folder}")

# 删除 SamplePDFFiles 文件夹及其内容
sample_folder = "SamplePDFFiles"
if os.path.exists(sample_folder):
    shutil.rmtree(sample_folder)
    print(f"Deleted folder: {sample_folder}")

## 3. 实训总结

通过本次实训，我们系统地学习了如何使用 `PyPDF2` 和 `pdfplumber` 两个强大的 Python 库来处理 PDF 文件。我们首先利用 `reportlab` 创建了多个示例 PDF 文件，包括文字内容和表格数据，为后续操作提供了素材。接着，我们使用 `PyPDF2` 完成了 PDF 文件的合并、拆分、加密和解密等操作，这些功能在文档管理和共享中非常实用。此外，我们还借助 `pdfplumber` 提取了 PDF 文件中的文本和表格数据，这对于数据分析和信息提取至关重要。通过这些实践操作，我们不仅掌握了 PDF 文件处理的基本技能，还了解了如何将这些技能应用于实际工作场景中，提高了工作效率和数据处理能力。